In [1]:
import psutil

ram_gb = psutil.virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')


import sys
print(sys.version)

print("\n---\nTransformers")
import transformers
print(transformers.__version__)

print("\n---\nTorch")
import torch
print("Compiled with cuda: ", torch.cuda.is_available())
print(torch.__version__)

# print("\n---\nBitsAndBytes")
# import bitsandbytes
# print(bitsandbytes.__version__)

# print(f"Compiled with CUDA: {bitsandbytes.cextension.lib.compiled_with_cuda == True}")


Your runtime has 404.3 gigabytes of available RAM

You are using a high-RAM runtime!
3.10.13 (main, Jul 23 2025, 16:24:01) [GCC 13.3.0]

---
Transformers


/tmp/igor.varejao/nesenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


5.8.0

---
Torch
Compiled with cuda:  True
2.11.0+cu126


In [2]:
tensor = torch.Tensor([1, 2, 3])
tensor = tensor.to('cuda:0')
tensor.device

device(type='cuda', index=0)

# Testing Inference

In [ ]:
MIDI_MODEL = "/storage/datasets/igor.varejao/checkpoints/checkpoint_midi_best.pth"



In [ ]:
def generate(
        dataset_class,
        dataset_root:str, 
        decoder_size:MusicGenSize,
        audio_secs:int=60,
        xp_dir = './'
    ):

    # Instanciate xp
    xp_folder, checkpoint_folder, generation_folder = get_xp(xp_dir)
    checkpoint_path = os.path.join(checkpoint_folder, f'checkpoint_best.pth')

    # Get models
    dac = DACCompressionModel.get_pretrained("44khz", device='cpu').eval()
    dac.set_num_codebooks(4)

    model = MusicGenTransformer(
        vocab_size = dac.vocab_size,
        frame_rate = int(dac.frame_rate),
        audio_duration = audio_secs,
        pad_token_id = dac.pad_token_id,
        eos_token_id = dac.eos_token_id,
        bos_token_id = dac.bos_token_id,
        model_size = decoder_size
    ).cuda().eval()


    logger.info(f"Loading checkpoint from {checkpoint_path}")
    ckpt = torch.load(checkpoint_path, weights_only=False)

    model.load_state_dict(ckpt["model"])
    logger.info(f"Loaded checkpoint from {checkpoint_path}")

    # Get dataloader
    test_dataset = dataset_class(
        dataset_root=dataset_root, 
        split="test",
        num_codebooks=model.num_codebooks,
        audio_secs=audio_secs
    )
    test_dataloader = DataLoader(
        test_dataset,
        batch_size=10,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        collate_fn=collate,
    )

    for batch in test_dataloader:
        file_name_prefix, input_tokens, target_tokens, meta, txt_tokens, txt_tokens_mask = batch

        for i in range(len(file_name_prefix)):
            conditioner = txt_tokens[i].unsqueeze(0).cuda()
            conditioner_mask = txt_tokens_mask[i].unsqueeze(0).cuda()

            # Run generation
            with torch.inference_mode():
                logger.info(f"Generating audio tokens for {file_name_prefix[i]}...")
                audio_tokens = model.generate(
                    max_new_tokens=model.max_seq_len,
                    src=conditioner,
                    src_mask=conditioner_mask,
                    temperature=2.0, # < 1, such as 1e-4 -> eliminate randomness | = 1 -> the distribution learned | > 1 -> aproximate a uniform distribution
                    cfg_scale=2.0,
                    top_k=250,
                    top_p=0.9
                )

                print(f"Generated audio codes shape: {audio_tokens.shape}\n")

                # Decode back to audio
                audio_tokens = torch.clamp(audio_tokens, min=0, max=dac.orig_vocab_size - 1)
                total_generated_samples = audio_tokens.shape[-1] * dac.model.hop_length
                meta[i]['original_length'] = min(meta[i]['original_length'], total_generated_samples)
                decoded_audio = dac.decode(audio_tokens.cpu(), meta[i])

                logger.info(f"Decoded audio shape: {decoded_audio.shape}")

                if not os.path.isdir(generation_folder):
                    os.makedirs(generation_folder)

                path = os.path.join(generation_folder, file_name_prefix[i]+'.wav')
                decoded_audio.write(path)

